# CutSceneAI guard scene: constrained Kimodo body motion

Runs one 12-second, 30 fps motion candidate from the **18fe** guard inputs. This is a body-motion experiment, not an approved canonical performance. Inspect the final motion before attempting engine import.

Use a Kaggle Notebook with **Accelerator: GPU T4 x2** and **Internet: On**. In **Add-ons → Secrets**, add a secret named `HF_TOKEN` whose Hugging Face account has approved access to `meta-llama/Meta-Llama-3-8B-Instruct`, and allow this notebook to read it. Keep the token out of notebook cells and chat messages. The second cell reads the secret securely; the last runs the pilot and saves `guard-kimodo-18fe-evidence.zip` in Outputs. Initial model download and native build can take time.


In [ ]:
import os, subprocess, sys
from pathlib import Path
print('Python:', sys.version.split()[0]); print('RAM:'); subprocess.run(['free', '-h'], check=True)
print('GPU:'); subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)


In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
if not os.environ['HF_TOKEN']:
    raise RuntimeError('Set and enable the HF_TOKEN Kaggle Secret first.')
print('Hugging Face credential available to this session (value hidden).')


In [ ]:
SCRIPT = '"""Run the 18fe guard body-motion pilot in a Kaggle GPU notebook.\n\nThis script intentionally keeps the private Hugging Face token out of the\nrepository and saves only motion evidence under /kaggle/working.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\n\n\nKIMODO_COMMIT = "58e781898b3d7e328a676a75d3e338c45dce3ad9"\nPILOT_DIR = Path(__file__).resolve().parent\nWORK_DIR = Path("/kaggle/working")\nOUT_DIR = WORK_DIR / "guard-kimodo-18fe"\nSRC_DIR = Path("/tmp/cutsceneai-kimodo-source")\n\n\ndef run(args: list[str], **kwargs: object) -> None:\n    print("Running:", " ".join(args), flush=True)\n    subprocess.run(args, check=True, **kwargs)\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef install_kimodo() -> None:\n    """Build MotionCorrection against Kaggle\'s Python rather than Ubuntu\'s 3.10."""\n    if os.geteuid() != 0:\n        raise RuntimeError("This notebook requires root to install C++ build dependencies.")\n    run(["apt-get", "update", "-qq"])\n    run(["apt-get", "install", "-y", "-qq", "cmake", "libeigen3-dev"])\n\n    # Ubuntu 22.04\'s pybind11-dev is 2.9.1. Its CMake package takes priority\n    # unless a Python-3.12-compatible package is put first on CMAKE_PREFIX_PATH.\n    run([sys.executable, "-m", "pip", "install", "pybind11==2.13.6"])\n    import pybind11\n\n    import sysconfig\n\n    header_dir = Path(sysconfig.get_path("include"))\n    if not (header_dir / "Python.h").is_file():\n        raise RuntimeError(f"Python {sys.version_info[:2]} headers missing at {header_dir}")\n\n    # A Python extension needs Development.Module, not Development.Embed.\n    # Ubuntu 22.04\'s python3-dev supplies 3.10 embed libraries while Kaggle\n    # runs 3.12, which can make CMake\'s full Development check fail.\n    cmake_file = SRC_DIR / "MotionCorrection" / "CMakeLists.txt"\n    original = "find_package(Python3 COMPONENTS Interpreter Development REQUIRED)"\n    replacement = "find_package(Python3 COMPONENTS Interpreter Development.Module REQUIRED)"\n    contents = cmake_file.read_text()\n    if original in contents:\n        cmake_file.write_text(contents.replace(original, replacement, 1))\n    elif replacement not in contents:\n        raise RuntimeError("Kimodo MotionCorrection CMake declaration changed; inspect upstream source.")\n\n    env = os.environ.copy()\n    env["CMAKE_PREFIX_PATH"] = os.pathsep.join(\n        filter(None, (pybind11.get_cmake_dir(), env.get("CMAKE_PREFIX_PATH", "")))\n    )\n    OUT_DIR.mkdir(parents=True, exist_ok=True)\n    logfile = OUT_DIR / "build.log"\n    print(f"Building Kimodo with pybind11 {pybind11.__version__}; log: {logfile}", flush=True)\n    with logfile.open("w", encoding="utf-8") as log:\n        result = subprocess.run(\n            [sys.executable, "-m", "pip", "install", "-e", str(SRC_DIR), "-v"],\n            env=env, stdout=log, stderr=subprocess.STDOUT, check=False,\n        )\n    if result.returncode:\n        lines = logfile.read_text(errors="replace").splitlines()\n        markers = [i for i, line in enumerate(lines) if re.search(\n            r"CMake Error|fatal error|Could NOT find|FAILED:|error: command", line\n        )]\n        if markers:\n            index = markers[0]\n            print("First native build error:\\n" + "\\n".join(lines[max(0, index - 8):index + 20]))\n        print("Last 35 build-log lines:\\n" + "\\n".join(lines[-35:]))\n        raise RuntimeError(f"Kimodo build failed; full log at {logfile}")\n    import motion_correction\n\n    print(f"Kimodo and MotionCorrection installed: {motion_correction.__file__}", flush=True)\n\n\ndef main() -> None:\n    if not WORK_DIR.is_dir():\n        raise RuntimeError("Run this pilot in a Kaggle notebook with a T4 GPU.")\n    token = os.environ.pop("HF_TOKEN", None)\n    if not token:\n        raise RuntimeError("Set HF_TOKEN via Kaggle Secrets; never paste it into notebook code.")\n\n    import torch\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select GPU T4 x2 in Kaggle Notebook Settings first.")\n    memory_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")\n    cgroup_limit = Path("/sys/fs/cgroup/memory.max")\n    if cgroup_limit.is_file() and cgroup_limit.read_text().strip().isdigit():\n        memory_bytes = min(memory_bytes, int(cgroup_limit.read_text().strip()))\n    memory_gib = memory_bytes / 1024**3\n    print(f"Python {sys.version.split()[0]}; PyTorch {torch.__version__}; GPU {torch.cuda.get_device_name(0)}; RAM {memory_gib:.1f} GiB")\n    if memory_gib < 24:\n        raise RuntimeError("The 8B CPU text encoder needs a notebook with about 29 GiB RAM.")\n\n    if not SRC_DIR.exists():\n        run(["git", "clone", "https://github.com/nv-tlabs/kimodo.git", str(SRC_DIR)])\n    run(["git", "-C", str(SRC_DIR), "fetch", "--depth", "1", "origin", KIMODO_COMMIT])\n    run(["git", "-C", str(SRC_DIR), "checkout", "--detach", KIMODO_COMMIT])\n\n    install_kimodo()\n\n    OUT_DIR.mkdir(parents=True, exist_ok=True)\n    for name in ("meta.json", "constraints.json"):\n        shutil.copy2(PILOT_DIR / name, OUT_DIR / name)\n    env = os.environ.copy()\n    env["HF_HOME"] = "/root/.cache/huggingface"  # outside Kaggle\'s downloadable outputs\n    env["TEXT_ENCODER_DEVICE"] = "cpu"\n    env["TEXT_ENCODER_MODE"] = "local"\n    env["HF_TOKEN"] = token\n    stem = OUT_DIR / "guard"\n    cmd = [\n        sys.executable, "-m", "kimodo.scripts.generate",\n        "--input_folder", str(PILOT_DIR),\n        "--model", "Kimodo-SOMA-RP-v1.1",\n        "--output", str(stem),\n        "--bvh", "--bvh_standard_tpose",\n    ]\n    logfile = OUT_DIR / "run.log"\n    print("Generating the 12-second guard motion; watch run.log if this takes a while.", flush=True)\n    with logfile.open("w", encoding="utf-8") as log:\n        result = subprocess.run(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, check=False)\n    if result.returncode:\n        print("Kimodo failed. Final log lines:")\n        print("\\n".join(logfile.read_text(errors="replace").splitlines()[-60:]))\n        raise RuntimeError(f"Kimodo exited with status {result.returncode}")\n\n    npz = stem.with_suffix(".npz")\n    bvh = stem.with_suffix(".bvh")\n    if not npz.is_file() or not bvh.is_file():\n        raise RuntimeError(f"Missing expected output: {npz} / {bvh}. Inspect {logfile}.")\n    import numpy as np\n\n    with np.load(npz) as motion:\n        arrays = {name: list(motion[name].shape) for name in motion.files}\n    manifest = {\n        "kimodo_commit": KIMODO_COMMIT,\n        "model": "nvidia/Kimodo-SOMA-RP-v1.1",\n        "native_build": "pybind11==2.13.6; CMake Python3 Development.Module",\n        "text_encoder_device": "cpu",\n        "torch": torch.__version__,\n        "gpu": torch.cuda.get_device_name(0),\n        "numpy_arrays": arrays,\n        "sha256": {p.name: sha256(p) for p in (\n            npz, bvh, logfile, OUT_DIR / "build.log", OUT_DIR / "meta.json", OUT_DIR / "constraints.json"\n        )},\n        "acceptance": "UNREVIEWED: asset creation alone does not pass visual acceptance",\n    }\n    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    archive = WORK_DIR / "guard-kimodo-18fe-evidence.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:\n        for path in sorted(OUT_DIR.iterdir()):\n            if path.is_file():\n                bundle.write(path, path.name)\n    print(f"Evidence ready: {archive}; arrays: {arrays}")\n    print("Download the ZIP and share it for visual review. Do not import to Unity yet.")\n\n\nif __name__ == "__main__":\n    main()\n'
META = '{\n  "texts": [\n    "A person walks cautiously forward through an empty hallway with natural, steady steps.",\n    "A person slows from a cautious walk, plants their feet, and listens after hearing a noise.",\n    "A person turns about forty degrees to the right toward a door, then holds an alert standing pose."\n  ],\n  "durations": [5.0, 2.0, 5.0],\n  "num_samples": 1,\n  "seed": 20260812,\n  "diffusion_steps": 100,\n  "cfg": {\n    "enabled": true,\n    "text_weight": 2.0,\n    "constraint_weight": 2.0\n  }\n}\n'
CONSTRAINTS = '[\n  {\n    "type": "root2d",\n    "frame_indices": [0, 30, 60, 90, 120, 149, 170, 180, 209, 225, 240, 300, 359],\n    "smooth_root_2d": [\n      [0.0, 0.0],\n      [0.0, 0.48],\n      [0.0, 0.96],\n      [0.0, 1.44],\n      [0.0, 1.92],\n      [0.0, 2.41],\n      [0.0, 2.58],\n      [0.0, 2.62],\n      [0.0, 2.62],\n      [0.0, 2.62],\n      [0.0, 2.62],\n      [0.0, 2.62],\n      [0.0, 2.62]\n    ],\n    "global_root_heading": [\n      [1.0, 0.0],\n      [1.0, 0.0],\n      [1.0, 0.0],\n      [1.0, 0.0],\n      [1.0, 0.0],\n      [1.0, 0.0],\n      [1.0, 0.0],\n      [1.0, 0.0],\n      [1.0, 0.0],\n      [0.939692621, 0.342020143],\n      [0.766044443, 0.642787610],\n      [0.766044443, 0.642787610],\n      [0.766044443, 0.642787610]\n    ]\n  }\n]\n'
import json
from pathlib import Path
input_dir = Path('/kaggle/working/cutsceneai-pilot-input')
input_dir.mkdir(parents=True, exist_ok=True)
(input_dir / 'kaggle_pilot.py').write_text(SCRIPT, encoding='utf-8')
(input_dir / 'meta.json').write_text(META, encoding='utf-8')
(input_dir / 'constraints.json').write_text(CONSTRAINTS, encoding='utf-8')
assert sum(json.loads((input_dir / 'meta.json').read_text())['durations']) == 12.0
print('Prepared the pinned 12-second guard inputs.')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(input_dir / 'kaggle_pilot.py')], check=True)
print('Download guard-kimodo-18fe-evidence.zip from notebook Outputs for visual review.')
